In [2]:
import pandas as pd
import sqlite3

# Load CSV files
cc = pd.read_csv(r'C:\Users\dell\Downloads\credit_card.csv')
customer = pd.read_csv(r'C:\Users\dell\Downloads\customer.csv')

# Check columns and shape
print("Credit Card columns:")
print(cc.columns.tolist())
print("Shape:", cc.shape)

print("\nCustomer columns:")
print(customer.columns.tolist())
print("Shape:", customer.shape)

print("\nCredit Card first 3 rows:")
print(cc.head(3))

print("\nCustomer first 3 rows:")
print(customer.head(3))

Credit Card columns:
['Client_Num', 'Card_Category', 'Annual_Fees', 'Activation_30_Days', 'Customer_Acq_Cost', 'Week_Start_Date', 'Week_Num', 'Qtr', 'current_year', 'Credit_Limit', 'Total_Revolving_Bal', 'Total_Trans_Amt', 'Total_Trans_Vol', 'Avg_Utilization_Ratio', 'Use Chip', 'Exp Type', 'Interest_Earned', 'Delinquent_Acc']
Shape: (10108, 18)

Customer columns:
['Client_Num', 'Customer_Age', 'Gender', 'Dependent_Count', 'Education_Level', 'Marital_Status', 'state_cd', 'Zipcode', 'Car_Owner', 'House_Owner', 'Personal_loan', 'contact', 'Customer_Job', 'Income', 'Cust_Satisfaction_Score']
Shape: (10108, 15)

Credit Card first 3 rows:
   Client_Num Card_Category  Annual_Fees  Activation_30_Days  \
0   708082083          Blue          200                   0   
1   708083283          Blue          445                   1   
2   708084558          Blue          140                   0   

   Customer_Acq_Cost Week_Start_Date Week_Num Qtr  current_year  Credit_Limit  \
0                 87 

In [5]:
# Create database
conn = sqlite3.connect(':memory:')
cc.to_sql('credit_card', conn, index=False)
customer.to_sql('customer', conn, index=False)

print("Database ready!")
print(f"Credit card rows: {len(cc)}")
print(f"Customer rows: {len(customer)}")

Database ready!
Credit card rows: 10108
Customer rows: 10108


In [7]:
# Check for Missing Values
q1 = pd.read_sql('''
    SELECT 
        SUM(CASE WHEN Card_Category IS NULL THEN 1 ELSE 0 END) as missing_card_category,
        SUM(CASE WHEN Credit_Limit IS NULL THEN 1 ELSE 0 END) as missing_credit_limit,
        SUM(CASE WHEN Total_Trans_Amt IS NULL THEN 1 ELSE 0 END) as missing_trans_amt,
        SUM(CASE WHEN Interest_Earned IS NULL THEN 1 ELSE 0 END) as missing_interest
    FROM credit_card
''', conn)
print("Missing values check:")
print(q1)

Missing values check:
   missing_card_category  missing_credit_limit  missing_trans_amt  \
0                      0                     0                  0   

   missing_interest  
0                 0  


In [8]:
#Revenue by Card Category
q2 = pd.read_sql('''
    SELECT 
        Card_Category,
        COUNT(*) as total_customers,
        ROUND(SUM(Annual_Fees), 2) as total_annual_fees,
        ROUND(SUM(Interest_Earned), 2) as total_interest,
        ROUND(SUM(Annual_Fees + Interest_Earned), 2) as total_revenue,
        ROUND(AVG(Credit_Limit), 2) as avg_credit_limit
    FROM credit_card
    GROUP BY Card_Category
    ORDER BY total_revenue DESC
''', conn)
print("Revenue by Card Category:")
print(q2)

Revenue by Card Category:
  Card_Category  total_customers  total_annual_fees  total_interest  \
0          Blue             9214          2685635.0      6495887.74   
1        Silver              639           187505.0       812081.28   
2          Gold              188            56210.0       373784.16   
3      Platinum               67            20665.0       161629.05   

   total_revenue  avg_credit_limit  
0     9181522.74           7285.66  
1      999586.28          23391.64  
2      429994.16          21857.84  
3      182294.05          16455.13  


In [10]:
#Weekly Revenue Trend
q3 = pd.read_sql('''
    SELECT 
        Week_Num,
        Qtr,
        COUNT(*) as transactions,
        ROUND(SUM(Total_Trans_Amt), 2) as weekly_transaction_amt,
        ROUND(SUM(Interest_Earned), 2) as weekly_interest,
        ROUND(SUM(Annual_Fees + Interest_Earned), 2) as weekly_revenue
    FROM credit_card
    GROUP BY Week_Num, Qtr
    ORDER BY Qtr, Week_Num
    LIMIT 10
''', conn)
print("Weekly Revenue Trend (first 10 weeks):")
print(q3)

Weekly Revenue Trend (first 10 weeks):
  Week_Num Qtr  transactions  weekly_transaction_amt  weekly_interest  \
0   Week-1  Q1           194                835767.0        143427.32   
1  Week-10  Q1           195                793080.0        138270.46   
2  Week-11  Q1           195                915725.0        155115.79   
3  Week-12  Q1           195                890081.0        156206.92   
4  Week-13  Q1           195                789941.0        131128.76   
5   Week-2  Q1           195                844739.0        150964.81   
6   Week-3  Q1           195                923367.0        167467.80   
7   Week-4  Q1           195                869235.0        144849.27   
8   Week-5  Q1           195                849078.0        158159.97   
9   Week-6  Q1           195                898867.0        163683.13   

   weekly_revenue  
0       199862.32  
1       194740.46  
2       215555.79  
3       216451.92  
4       188623.76  
5       208349.81  
6       224882.80

In [11]:
#join Tables — Customer + Card Data
q4 = pd.read_sql('''
    SELECT 
        c.Client_Num,
        cu.Customer_Age,
        cu.Gender,
        cu.Customer_Job,
        cu.Income,
        c.Card_Category,
        c.Credit_Limit,
        c.Total_Trans_Amt,
        c.Interest_Earned,
        c.Delinquent_Acc
    FROM credit_card c
    JOIN customer cu ON c.Client_Num = cu.Client_Num
    LIMIT 10
''', conn)
print("Joined table (first 10 rows):")
print(q4)

Joined table (first 10 rows):
   Client_Num  Customer_Age Gender   Customer_Job  Income Card_Category  \
0   708082083            24      F    Businessman  202326          Blue   
1   708083283            62      F  Selfemployeed    5225          Blue   
2   708084558            32      F  Selfemployeed   14235          Blue   
3   708085458            38      M    Blue-collar   45683          Blue   
4   708086958            48      M    Businessman   59279          Blue   
5   708095133            33      F  Selfemployeed   14254          Blue   
6   708098133            34      F  Selfemployeed   14975          Blue   
7   708099183            34      F       Retirees   31982          Blue   
8   708100533            48      M    Businessman   86668          Blue   
9   708103608            53      F    Businessman  223196      Platinum   

   Credit_Limit  Total_Trans_Amt  Interest_Earned  Delinquent_Acc  
0        3544.0            15149          4393.21               0  
1       

In [14]:
#Revenue by Customer Job
q5 = pd.read_sql('''
    SELECT 
        cu.Customer_Job,
        COUNT(*) as total_customers,
        ROUND(SUM(c.Annual_Fees + c.Interest_Earned), 2) as total_revenue,
        ROUND(AVG(c.Credit_Limit), 2) as avg_credit_limit,
        SUM(c.Delinquent_Acc) as total_delinquent
    FROM credit_card c
    JOIN customer cu ON c.Client_Num = cu.Client_Num
    GROUP BY cu.Customer_Job
    ORDER BY total_revenue DESC
''', conn)
print("Revenue by Customer Job:")
print(q5)

Revenue by Customer Job:
    Customer_Job  total_customers  total_revenue  avg_credit_limit  \
0    Businessman             1901     3102420.31           9896.10   
1   White-collar             1542     1892913.93           8904.68   
2  Selfemployeed             2575     1866731.81           8509.77   
3           Govt             1525     1603826.37           8117.62   
4    Blue-collar             1579     1415440.57           8021.67   
5       Retirees              986      912064.24           7897.88   

   total_delinquent  
0               101  
1                85  
2               167  
3               113  
4                87  
5                61  


In [15]:
#Age Group Analysis

q6 = pd.read_sql('''
    SELECT 
        CASE 
            WHEN cu.Customer_Age < 30 THEN '18-29'
            WHEN cu.Customer_Age < 40 THEN '30-39'
            WHEN cu.Customer_Age < 50 THEN '40-49'
            WHEN cu.Customer_Age < 60 THEN '50-59'
            ELSE '60+' 
        END as age_group,
        COUNT(*) as total_customers,
        ROUND(SUM(c.Total_Trans_Amt), 2) as total_transactions,
        ROUND(SUM(c.Annual_Fees + c.Interest_Earned), 2) as total_revenue,
        ROUND(AVG(c.Avg_Utilization_Ratio), 3) as avg_utilization
    FROM credit_card c
    JOIN customer cu ON c.Client_Num = cu.Client_Num
    GROUP BY age_group
    ORDER BY total_revenue DESC
''', conn)
print("Revenue by Age Group:")
print(q6)

Revenue by Age Group:
  age_group  total_customers  total_transactions  total_revenue  \
0     40-49             4529          19513415.0     4770082.67   
1     50-59             2993          14715311.0     3473454.09   
2     30-39             1839           7701120.0     1880779.16   
3       60+              532           1753122.0      455815.12   
4     18-29              215            839045.0      213266.19   

   avg_utilization  
0            0.279  
1            0.268  
2            0.276  
3            0.280  
4            0.270  


In [16]:
#Income Group Analysis
q7 = pd.read_sql('''
    SELECT 
        CASE 
            WHEN cu.Income < 20000 THEN 'Low Income'
            WHEN cu.Income < 50000 THEN 'Medium Income'
            WHEN cu.Income < 100000 THEN 'High Income'
            ELSE 'Very High Income'
        END as income_group,
        COUNT(*) as total_customers,
        ROUND(SUM(c.Annual_Fees + c.Interest_Earned), 2) as total_revenue,
        ROUND(AVG(c.Credit_Limit), 2) as avg_credit_limit,
        ROUND(AVG(c.Avg_Utilization_Ratio), 3) as avg_utilization
    FROM credit_card c
    JOIN customer cu ON c.Client_Num = cu.Client_Num
    GROUP BY income_group
    ORDER BY total_revenue DESC
''', conn)
print("Revenue by Income Group:")
print(q7)

Revenue by Income Group:
       income_group  total_customers  total_revenue  avg_credit_limit  \
0  Very High Income             1484     3428004.08          14322.29   
1       High Income             3101     3395086.64           5611.75   
2     Medium Income             3312     2746127.05           8486.61   
3        Low Income             2211     1224179.46           9283.18   

   avg_utilization  
0            0.175  
1            0.354  
2            0.248  
3            0.271  


In [17]:
#Delinquent Accounts Analysis
q8 = pd.read_sql('''
    SELECT 
        cu.Customer_Job,
        cu.Education_Level,
        COUNT(*) as total_customers,
        SUM(c.Delinquent_Acc) as delinquent_count,
        ROUND(SUM(c.Delinquent_Acc) * 100.0 / COUNT(*), 1) as delinquent_rate_pct
    FROM credit_card c
    JOIN customer cu ON c.Client_Num = cu.Client_Num
    GROUP BY cu.Customer_Job, cu.Education_Level
    HAVING delinquent_count > 0
    ORDER BY delinquent_rate_pct DESC
    LIMIT 10
''', conn)
print("Top Delinquent Groups:")
print(q8)

Top Delinquent Groups:
    Customer_Job Education_Level  total_customers  delinquent_count  \
0  Selfemployeed       Doctorate              131                15   
1           Govt      Uneducated              212                20   
2           Govt         Unknown              249                23   
3           Govt       Doctorate               70                 6   
4       Retirees         Unknown              151                13   
5  Selfemployeed   Post-Graduate              134                11   
6    Blue-collar         Unknown              231                18   
7       Retirees     High School              188                14   
8           Govt        Graduate              601                44   
9   White-collar      Uneducated              207                15   

   delinquent_rate_pct  
0                 11.5  
1                  9.4  
2                  9.2  
3                  8.6  
4                  8.6  
5                  8.2  
6                  7

In [18]:
#Top Spending States
q9 = pd.read_sql('''
    SELECT 
        cu.state_cd,
        COUNT(*) as total_customers,
        ROUND(SUM(c.Total_Trans_Amt), 2) as total_spending,
        ROUND(AVG(c.Total_Trans_Amt), 2) as avg_spending,
        ROUND(SUM(c.Annual_Fees + c.Interest_Earned), 2) as total_revenue
    FROM credit_card c
    JOIN customer cu ON c.Client_Num = cu.Client_Num
    GROUP BY cu.state_cd
    ORDER BY total_revenue DESC
    LIMIT 10
''', conn)
print("Top 10 States by Revenue:")
print(q9)

Top 10 States by Revenue:
  state_cd  total_customers  total_spending  avg_spending  total_revenue
0       TX             2394      10314288.0       4308.39     2495012.67
1       CA             2468      10121249.0       4100.99     2494025.39
2       NY             2270      10258352.0       4519.10     2475102.75
3       FL             1711       7797286.0       4557.15     1893661.18
4       NJ              716       3443779.0       4809.75      801662.70
5       IL               58        294908.0       5084.62       69443.85
6       MI               63        293540.0       4659.37       68607.97
7       IA               47        234732.0       4994.30       56988.47
8       NV               56        203548.0       3634.79       55600.05
9       VA               27        155266.0       5750.59       33632.31


In [19]:
#Export Clean Joined Dataset
q10 = pd.read_sql('''
    SELECT 
        c.*,
        cu.Customer_Age,
        cu.Gender,
        cu.Education_Level,
        cu.Customer_Job,
        cu.Income,
        cu.Marital_Status,
        cu.state_cd,
        cu.Cust_Satisfaction_Score,
        CASE 
            WHEN cu.Customer_Age < 30 THEN '18-29'
            WHEN cu.Customer_Age < 40 THEN '30-39'
            WHEN cu.Customer_Age < 50 THEN '40-49'
            WHEN cu.Customer_Age < 60 THEN '50-59'
            ELSE '60+' 
        END as Age_Group,
        CASE 
            WHEN cu.Income < 20000 THEN 'Low Income'
            WHEN cu.Income < 50000 THEN 'Medium Income'
            WHEN cu.Income < 100000 THEN 'High Income'
            ELSE 'Very High Income'
        END as Income_Group
    FROM credit_card c
    JOIN customer cu ON c.Client_Num = cu.Client_Num
''', conn)

# Export for Power BI
q10.to_csv('credit_card_clean.csv', index=False)
print(f"Clean dataset exported!")
print(f"Rows: {len(q10)}")
print(f"Columns: {q10.columns.tolist()}")

Clean dataset exported!
Rows: 10108
Columns: ['Client_Num', 'Card_Category', 'Annual_Fees', 'Activation_30_Days', 'Customer_Acq_Cost', 'Week_Start_Date', 'Week_Num', 'Qtr', 'current_year', 'Credit_Limit', 'Total_Revolving_Bal', 'Total_Trans_Amt', 'Total_Trans_Vol', 'Avg_Utilization_Ratio', 'Use Chip', 'Exp Type', 'Interest_Earned', 'Delinquent_Acc', 'Customer_Age', 'Gender', 'Education_Level', 'Customer_Job', 'Income', 'Marital_Status', 'state_cd', 'Cust_Satisfaction_Score', 'Age_Group', 'Income_Group']
